# Absolute Mitochondrial Heterogeneity Index (AMHI)

This notebook demonstrates all functions of the AMHI module.

## What this gives you

The existing **MHI-D** is *relative*: it refits its PCA space on whatever subjects are present, so adding one new subject shifts all previous values. Nothing is comparable across experiments.

The **AMHI** module fixes this by fitting the transform pipeline **once** on a reference population and freezing it. Any future subject — even from a different experiment — is projected into the same coordinate system.

### Three complementary absolute metrics

| Metric | Answers | Subject A (tight, far from AllMitoMean) | Subject B (dispersed, near AllMitoMean) |
|---|---|---|---|
| `MHI_D_absolute` | How heterogeneous is this subject internally? | **LOW** | **HIGH** |
| `AMHI_offset` | How different is this subject's typical mito from the global reference? | **HIGH** | **LOW** |
| `AMHI_D` | Total per-mito deviation from AllMitoMean (offset + spread combined) | varies | varies |

Mathematical relationship: `AMHI_D_mean² ≈ AMHI_offset² + MHI_D_absolute_mean²`

### Pipeline

```
1. compute_all_mito_mean()    → freeze reference (run once per study)
2. plot_radar()               → visualise AllMitoMean profile
3. compute_amhi()             → absolute position relative to AllMitoMean
4. compute_mhi_d_absolute()   → absolute internal heterogeneity
5. plot_amhi_distances()      → violin distributions + AMHI_offset overlay
6. plot_amhi_2d()             → 2D scatter, AllMitoMean at centre
```

## 0 — Imports

In [ ]:
import anndata

from mito_marker import (
    ingest_tem_folder,
    # Reference pipeline
    compute_all_mito_mean,
    # Absolute metrics
    compute_amhi,
    compute_mhi_d_absolute,
    # Plots
    plot_amhi_distances,
    plot_amhi_2d,
    plot_radar,
)

## 1 — Load TEM data

Adapt `data_directory_path` to your environment:
- **Colab**: `"/content/drive/MyDrive/mito-marker/data/raw/tem/"`
- **Codespaces**: `"data/raw/tem/"`

In [ ]:
tem_anndata = ingest_tem_folder(data_directory_path="data/raw/tem/")

print(f"Loaded: {tem_anndata.n_obs} mitochondria × {tem_anndata.n_vars} features")
print(f"Species: {tem_anndata.obs['specie'].value_counts().to_dict()}")
print(f"Conditions: {tem_anndata.obs['condition'].value_counts().to_dict()}")
print(f"Subjects: {tem_anndata.obs['unique_subject_ID'].nunique()}")

## 2 — Compute AllMitoMean and freeze the reference space

`compute_all_mito_mean()` does three things:
1. Fits a **StandardScaler** (Z-score) and **PCA** (95% variance) on the chosen reference population
2. Stores both fitted objects in `.uns['amhi_reference']` — they are never changed again
3. Injects **AllMitoMean** as a synthetic observation row (`unique_subject_ID = 'AllMitoMean'`)

### Option A: all subjects define the reference (default)

In [ ]:
tem_anndata, all_mito_mean_vector = compute_all_mito_mean(
    tem_anndata,
    obs_filter=None,           # all subjects
    n_pca_components="auto",   # auto-select components for 95% variance
    add_to_anndata=True,       # inject AllMitoMean as a synthetic obs row
    reference_label="AllMitoMean",
)

print(f"Reference mean vector shape: {all_mito_mean_vector.shape}")
print(f"AnnData now has {tem_anndata.n_obs} rows (original mitos + 1 synthetic AllMitoMean)")

### Option B: only mammals define the reference

The `obs_filter` restricts the population used to **fit** the transforms and compute the mean.
All other subjects remain in the AnnData and can be measured against this reference.

In [ ]:
# Reload clean data to demonstrate the filter option independently
tem_anndata_mammals_ref = ingest_tem_folder(data_directory_path="data/raw/tem/")

tem_anndata_mammals_ref, mammal_mean_vector = compute_all_mito_mean(
    tem_anndata_mammals_ref,
    obs_filter={"specie": ["Human", "Mouse"]},   # only mammals define the reference
    reference_label="MammalMean",
)

print(f"Reference based on {tem_anndata_mammals_ref.uns['amhi_reference']['n_obs_used']} mitos")
print(f"Species in reference: {tem_anndata_mammals_ref.uns['amhi_reference']['species_included']}")

### Option C: only young subjects define the reference

In [ ]:
tem_anndata_young_ref = ingest_tem_folder(data_directory_path="data/raw/tem/")

tem_anndata_young_ref, young_mean_vector = compute_all_mito_mean(
    tem_anndata_young_ref,
    obs_filter={"condition": "Young"},
    reference_label="YoungMean",
)

print(f"Young reference: {tem_anndata_young_ref.uns['amhi_reference']['n_obs_used']} mitos")

# Continue the demo with the all-subjects reference (Option A)
# tem_anndata already has AllMitoMean frozen from cell 2.1

## 3 — Inspect what is stored in .uns

The frozen transforms live here. You can save this AnnData and reload it months later — the reference is preserved.

In [ ]:
ref = tem_anndata.uns["amhi_reference"]

print("Keys in .uns['amhi_reference']:")
for key, value in ref.items():
    if key in ("scaler_bytes", "pca_bytes"):
        print(f"  {key}: <{len(value)} bytes pickled object>")
    else:
        print(f"  {key}: {value}")

## 4 — Radar plot of AllMitoMean alongside other subjects

Because AllMitoMean was injected as a real obs row, `plot_radar()` includes it automatically.
No special parameters needed.

In [ ]:
# AllMitoMean appears as one of the groups — it should sit near the centre
# of all other subject profiles (it is their average)
plot_radar(
    tem_anndata,
    group_by="unique_subject_ID",
    title="Mito feature profiles — AllMitoMean vs individual subjects",
)

In [ ]:
# Compare AllMitoMean vs condition groups (Young / Old)
plot_radar(
    tem_anndata,
    group_by="condition",
    title="Mito feature profiles — AllMitoMean vs conditions",
)

## 5 — Compute AMHI (absolute position relative to AllMitoMean)

`compute_amhi()` uses the **frozen** transforms. It never refits.

Returns per group:
- `AMHI_offset` — distance of the group's centroid to AllMitoMean
- `AMHI_D_median` — median per-mito distance to AllMitoMean (offset + spread combined)

In [ ]:
# Per subject
amhi_per_subject = compute_amhi(
    tem_anndata,
    group_by="unique_subject_ID",
)
# Results are printed automatically. Also stored in .uns['amhi_results']

In [ ]:
# Per species
amhi_per_species = compute_amhi(
    tem_anndata,
    group_by="specie",
)

In [ ]:
# Per condition (Young / Old)
amhi_per_condition = compute_amhi(
    tem_anndata,
    group_by="condition",
)

In [ ]:
# Measure only one subset — obs_filter is independent of what built the reference
# Example: measure only KFish, even though the reference includes all species
amhi_kfish_only = compute_amhi(
    tem_anndata,
    group_by="unique_subject_ID",
    obs_filter={"specie": "KFish"},
)

## 6 — Compute MHI_D_absolute (absolute internal heterogeneity)

Same formula as MHI-D (median distance to own centroid) but computed in the **frozen** reference space.

- Subject A (tight cluster, even if far from AllMitoMean) → **low** MHI_D_absolute
- Subject B (dispersed cluster, even if close to AllMitoMean) → **high** MHI_D_absolute

Adding a new subject later does **not** change any existing subject's MHI_D_absolute value.

In [ ]:
# Per subject
mhid_absolute_per_subject = compute_mhi_d_absolute(
    tem_anndata,
    group_by="unique_subject_ID",
)

In [ ]:
# Per species
mhid_absolute_per_species = compute_mhi_d_absolute(
    tem_anndata,
    group_by="specie",
)

### Compare AMHI_offset vs MHI_D_absolute

These two numbers capture orthogonal properties. Looking at them side by side is the key analytical read.

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "AMHI_offset": amhi_per_subject["AMHI_offset"],
    "MHI_D_absolute_median": mhid_absolute_per_subject["MHI_D_absolute_median"],
    "AMHI_D_median": amhi_per_subject["AMHI_D_median"],
    "n_mitos": amhi_per_subject["n_mitos"],
}).sort_values("AMHI_offset", ascending=False)

print("\nFull comparison (sorted by AMHI_offset):")
print(comparison.to_string())

print("\nInterpretation:")
print("  AMHI_offset      — how far the subject's typical mito is from the global reference")
print("  MHI_D_absolute   — how heterogeneous the subject's mito population is internally")
print("  AMHI_D           — total deviation from AllMitoMean (offset + spread combined)")

## 7 — Plot AMHI distances (violin + AMHI_offset overlay)

- **Violins** show the full per-mito AMHI_D distribution
- **Red dots** show AMHI_offset (distance of the group centroid to AllMitoMean)

When the red dot is close to the violin centre: most of the total AMHI_D comes from internal spread.
When the red dot is high relative to the violin centre: most of the AMHI_D comes from offset.

In [ ]:
# Per subject — see individual heterogeneity
plot_amhi_distances(
    tem_anndata,
    group_by="unique_subject_ID",
    title="AMHI distances per subject",
)

In [ ]:
# Per species — cross-species comparison
plot_amhi_distances(
    tem_anndata,
    group_by="specie",
    title="AMHI distances per species",
)

In [ ]:
# Per condition (Young / Old) — aging effect
plot_amhi_distances(
    tem_anndata,
    group_by="condition",
    title="AMHI distances — Young vs Old",
)

In [ ]:
# Pass pre-computed results to skip recomputation
plot_amhi_distances(
    tem_anndata,
    group_by="unique_subject_ID",
    amhi_results=amhi_per_subject,   # reuse already-computed DataFrame
    title="AMHI distances per subject (pre-computed)",
)

## 8 — Plot AMHI 2D scatter (AllMitoMean at centre)

Each dot = one mitochondrion. AllMitoMean is the **gold star at (0, 0)**.

- **Radial distance from centre** = individual mito's AMHI_D
- **Position of the cluster centre** = AMHI_offset
- **Spread of the cluster** = MHI_D_absolute

Concentric grey rings mark 1× and 2× the population median AMHI_D.

In [ ]:
# Colour by condition (Young / Old)
plot_amhi_2d(
    tem_anndata,
    color_by="condition",
    title="AMHI 2D — coloured by condition",
)

In [ ]:
# Colour by species
plot_amhi_2d(
    tem_anndata,
    color_by="specie",
    title="AMHI 2D — coloured by species",
)

In [ ]:
# Colour by individual subject
plot_amhi_2d(
    tem_anndata,
    color_by="unique_subject_ID",
    max_points_per_group=200,   # subsample large subjects for clarity
    title="AMHI 2D — coloured by subject",
)

In [ ]:
# Colour by combined label: species × condition
plot_amhi_2d(
    tem_anndata,
    color_by=["specie", "condition"],   # list → combined label "KFish / Young" etc.
    title="AMHI 2D — coloured by species × condition",
)

In [ ]:
# Show only one species — obs_filter restricts what is plotted
plot_amhi_2d(
    tem_anndata,
    color_by="condition",
    obs_filter={"specie": "KFish"},
    title="AMHI 2D — KFish only, coloured by condition",
)

In [ ]:
# Without distance rings (cleaner for publication)
plot_amhi_2d(
    tem_anndata,
    color_by="condition",
    show_distance_rings=False,
    title="AMHI 2D — no distance rings",
)

## 9 — Cross-experiment workflow: adding a new subject

This is the core use case for the absolute index.
Once the reference is frozen, a new subject can be measured on the **same scale** as all existing subjects.

In [ ]:
# Step 1: save the AnnData with frozen transforms
tem_anndata.write_h5ad("data/processed/tem_with_amhi_reference.h5ad")
print("AnnData saved. Transforms are frozen inside .uns['amhi_reference'].")

In [ ]:
# Step 2 (later, in a different session): reload and ingest the new subject
frozen_anndata = anndata.read_h5ad("data/processed/tem_with_amhi_reference.h5ad")

new_subject_anndata = ingest_tem_folder(data_directory_path="data/raw/tem/new_subject/")

# Step 3: concatenate — transforms are NOT touched, only obs rows are added
combined_anndata = anndata.concat(
    [frozen_anndata, new_subject_anndata],
    merge="same",
)
combined_anndata.uns = dict(frozen_anndata.uns)   # carry over frozen transforms

print(f"Combined: {combined_anndata.n_obs} mitochondria")

In [ ]:
# Step 4: compute metrics — old subjects are IDENTICAL to before (fixed space)
amhi_combined = compute_amhi(combined_anndata, group_by="unique_subject_ID")
mhid_combined = compute_mhi_d_absolute(combined_anndata, group_by="unique_subject_ID")

print("\nNew subject's metrics on the same absolute scale as all existing subjects:")
print(amhi_combined.tail(3).to_string())

In [ ]:
# Step 5: visualise the new subject in context
plot_amhi_2d(
    combined_anndata,
    color_by="unique_subject_ID",
    title="AMHI 2D — original subjects + new subject",
)

## 10 — Verify the decomposition identity

`AMHI_D_mean² ≈ AMHI_offset² + MHI_D_absolute_mean²`

This is the bias-variance decomposition. It holds for means (not medians) and shows that
AMHI_D combines offset and internal spread without double-counting.

In [ ]:
import numpy as np

print(f"{'Subject':<25} {'AMHI_D²':>10} {'offset²+MHI²':>14} {'error %':>10}")
print("-" * 62)

for subject in amhi_per_subject.index:
    amhi_d_sq = amhi_per_subject.loc[subject, "AMHI_D_mean"] ** 2
    offset_sq = amhi_per_subject.loc[subject, "AMHI_offset"] ** 2
    mhid_sq   = mhid_absolute_per_subject.loc[subject, "MHI_D_absolute_mean"] ** 2
    decomposed = offset_sq + mhid_sq
    error_pct  = abs(amhi_d_sq - decomposed) / (amhi_d_sq + 1e-9) * 100
    print(f"{subject:<25} {amhi_d_sq:>10.4f} {decomposed:>14.4f} {error_pct:>9.1f}%")

print("\n(Small error % confirms the decomposition holds for means)")

## 11 — Using a subset-based reference: measure all species against a Young-only baseline

Scientific question: *How far has each species drifted from the Young mito phenotype?*

In [ ]:
# Use the Young-reference AnnData computed in cell 2.3
# Measure ALL species against the Young reference
amhi_vs_young = compute_amhi(
    tem_anndata_young_ref,
    group_by="specie",
    reference_label="YoungMean",
)

print("\nHow far is each species from the Young reference mito?")
print(amhi_vs_young[["n_mitos", "AMHI_offset", "AMHI_D_median"]].sort_values(
    "AMHI_offset", ascending=False
).to_string())

In [ ]:
plot_amhi_2d(
    tem_anndata_young_ref,
    color_by="condition",
    reference_label="YoungMean",
    title="AMHI 2D — reference = Young subjects only",
)

In [ ]:
plot_amhi_distances(
    tem_anndata_young_ref,
    group_by="condition",
    reference_label="YoungMean",
    title="AMHI distances — how far is each condition from the Young reference?",
)